# Preprocessing pipeline

In [17]:
#imports
import sys
sys.path.append('/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope')

from preprocessing import DataLoader

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import librosa.display
import os
import librosa as lb
import soundfile as sf

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# Loading data -> all factors dataframe

Demographic data
  - Patient number
  - Age
  - Sex
  - Adult BMI (kg/m2)
  - Child Weight (kg)
  - Child Height (cm)

Patient data

Audio text files

In [ ]:
#define paths
audio_path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/',
demographic_path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/demographic_info.txt',
diagnosis_path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv'

#instantiate DataLoader
loader = DataLoader(audio_path, demographic_path, diagnosis_path)

# Cleaning

-blanks

In [10]:
#block to complete

#imputing as per keira

# Preprocessing
-encoding
-standardising

In [8]:
#dealing with numericals

#one hot encode disease, equipment, acquisition mode, chest location, recording_index

cols = ['disease', 'recording_index', 'chest_location', 'acquisition_mode']
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(allfactors_data[cols])

encoded_df = pd.DataFrame(encoded,
                          columns=encoder.get_feature_names_out(cols),
                          index=allfactors_data.index)

allfactors_data = pd.concat([allfactors_data, encoded_df], axis=1).drop(columns=cols)

#binary code gender
le = LabelEncoder()
allfactors_data['sex'] = allfactors_data['sex'].map({'M': 0, 'F': 1})
allfactors_data.head()


,start,end,crackles,weezels,pid,equipment,filename,age,sex,adult_bmi,...,recording_index_8p3,chest_location_Al,chest_location_Ar,chest_location_Ll,chest_location_Lr,chest_location_Pl,chest_location_Pr,chest_location_Tc,acquisition_mode_mc,acquisition_mode_sc
0,0.022,0.364,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.364,2.436,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2.436,4.636,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4.636,6.793,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,6.793,8.750,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


# Features engineering
-one BMI column

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit

# run pre-split pipeline
pre_split_pipeline = Pipeline([
    ('load', DataLoader(
        audio_path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/',
        demographic_path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/demographic_info.txt',
        diagnosis_path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv'
    )),
    ('encode', FeatureEncoder()),
    ('bmi', BMICalculator()),
])

data = pre_split_pipeline.fit_transform(None)

# separate X and y
disease_cols = [col for col in data.columns if col.startswith('disease_')]
X = data.drop(columns=disease_cols)
y = data[disease_cols]

# patient-level split
groups = X['pid']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# check class distribution
print("Train class distribution:")
print(y_train.sum())
print("\nTest class distribution:")
print(y_test.sum())

NameError: name 'FeatureEncoder' is not defined